# RAGAS Evaluation: Spinal Health RAG Pipeline
*Phase 2 of MedRAG-Eval — automated evaluation using RAGAS metrics*

*For pipeline details see 01_rag_pipeline.ipynb*

In [1]:
pip install ragas==0.2.14 -q
!pip install langchain-google-vertexai -q
!pip install numpy==1.26.4 --only-binary=:all: -q

# Base packages for this section
!pip install -qU langchain-core langchain-text-splitters langchain-community langchain-chroma langchain-ollama beautifulsoup4 lxml

# If you hit: deepeval requires click<8.4.0 but click 8.4.1 is installed
# run the three commands below to fix and verify dependency conflicts.
!python -m pip install "click>=8.0.0,<8.4.0"
!python -m pip install --upgrade --force-reinstall "deepeval==4.0.4"
!python -m pip check

from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama


llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:7b",
    temperature=0,
    max_tokens = 200
)

# Load data from Web
loader = WebBaseLoader(["https://www.webmd.com/cancer/what-is-chordoma",
                        "https://www.webmd.com/brain/spinal-muscular-atrophy",
                        "https://www.webmd.com/back-pain/spinal-stenosis",
                        "https://www.webmd.com/pain-management/pain-management-treatment-overview"])
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever(
    search_kwargs={"k": 10}
)

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}

    Be accurate and specific. If the answer is not in the context, say you don't know.

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()

In [2]:
import sys
import types

vertexai_chat_module = types.ModuleType("langchain_community.chat_models.vertexai")
vertexai_chat_module.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = vertexai_chat_module

vertexai_llm_module = types.ModuleType("langchain_community.llms.vertexai")
vertexai_llm_module.VertexAI = type("VertexAI", (), {})
sys.modules["langchain_community.llms.vertexai"] = vertexai_llm_module



0.2.14


## Test Dataset

14 factual questions across 4 spinal health topics, verified by a spinal orthopedic specialist.
Ground truth answers sourced directly from WebMD articles used in the knowledge base.

**Coverage:**
- Chordoma — 3 questions
- Spinal Muscular Atrophy (SMA) — 4 questions  
- Spinal Stenosis — 4 questions
- Pain Management — 3 questions

In [3]:
data_list = []

test_cases = [
    {"question": "which cells chordoma develops from?",
     "ground_truth": "notochord cells"},
    {"question": "which symtoms will patient have with chordoma compressing spinal cord in thoracic spine?",
     "ground_truth": "Numbness, tingling, or weakness in your legs, loss of bowel or bladder control"},
    {"question": "why doctors may order bone scan and lungs ct as part of workup of chordoma?",
     "ground_truth": "to check for potential spread of the tumor"},
    {"question": "which gene problem causes SMA?",
     "ground_truth": "SMN1"},
    {"question": "what happens with spinal cord without SMN protein?",
     "ground_truth": "spinal cord can't send strong signals to control   muscles. then  the muscle cells  slowly die"},
    {"question": "how SMA usually gets diagnosed in USA?",
     "ground_truth": "SMA is part of the recommended uniform screening Panel and is performed at birth"},
    {"question": "which gene therapy drugs are approved by fda in USA?",
     "ground_truth": "nusinersen (Spinraza), onasemnogene abeparvovec-xioi (Zolgensma), and risdiplam (Evrysdi)"},
    {"question": "which types of spinal stenosis there are?",
     "ground_truth": "cervical, lumbar, thoracic"},
    {"question": "what usually causes narrowing of spinal canal?",
     "ground_truth": "bone spurs and thickened ligaments"},
    {"question": "which symptom comes last, when the most severe spinal stenosis develops?",
     "ground_truth": "bowel and bladder control loss"},
    {"question": "what include laminectomy surgery?",
     "ground_truth": "removal of posterior part of the vertebral canal"},
    {"question": "what is the main downside of using NSAIDS?",
     "ground_truth": "risk of heart attack or stroke"},
    {"question": "For which type of pain doctors use  steroid injections to trigger points?",
     "ground_truth": "muscle pain and chronic pain involving tissue that surrounds muscle"},
    {"question": "what is the main advantage of intrathecal drug delivery?",
     "ground_truth": "less dose of pain medications needed and due to this patient will have less side effects"}

]

data = {
        "question": [],
        "answer": [],
        "contexts": [],
        "ground_truth": []
}

for tc in test_cases:
    docs = retriever.invoke(tc["question"])
    data["question"].append(tc["question"])
    data["contexts"].append([doc.page_content for doc in docs])
    data["answer"].append(chain.invoke(tc["question"]))
    data["ground_truth"].append(tc["ground_truth"])

data_list = [
    {
        "user_input": data["question"][i],
        "response": data["answer"][i],
        "retrieved_contexts": data["contexts"][i],
        "reference": data["ground_truth"][i]
    }
    for i in range(len(data["question"]))
]



## Evaluation Setup

Importing RAGAS metrics and configuring local LLM as evaluator.

**Evaluator model:** `qwen2.5:7b` — same model used for generation, 
ensuring metric differences in Phase 3 reflect framework behavior, not model differences.

**Known issue:** `LangchainLLMWrapper` is deprecated in RAGAS 0.4.x. 
Pinned to `ragas==0.2.14` for Ollama compatibility.

In [4]:

from ragas.metrics import ContextPrecision, ContextRecall, NoiseSensitivity, AnswerRelevancy, Faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas import EvaluationDataset, evaluate
from ragas.run_config import RunConfig
from ragas.embeddings import LangchainEmbeddingsWrapper

import nest_asyncio
nest_asyncio.apply()

config = RunConfig(
    timeout=9000.0,       # Maximum time (in seconds) to wait for a single operation (Default is 180)
    max_workers=1
)


evaluator_embeddings = LangchainEmbeddingsWrapper(embedding)

evaluator_llm = LangchainLLMWrapper(llm)

evaluation_dataset = EvaluationDataset.from_list(data_list)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        ContextPrecision(llm=evaluator_llm),
        ContextRecall(llm=evaluator_llm),
        NoiseSensitivity(llm=evaluator_llm),
        AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
        Faithfulness(llm=evaluator_llm)
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=config
)





Evaluating:   0%|          | 0/70 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\artom\miniconda3\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002370D13AAC0> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-4' coro=<_async_in_context.<locals>.run_in_context() running at C:\Users\artom\miniconda3\Lib\site-packages\ipykernel\utils.py:60> wait_for=<Task pending name='Task-6' coro=<Kernel.shell_main() running at C:\Users\artom\miniconda3\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\artom\miniconda3\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\artom\miniconda3\Lib\site-packages\anyio\abc\_sockets.py:141: RuntimeWarning: coroutine 'Kernel.shell_main

## Results

In [9]:
df = result.to_pandas()

print(df.columns.tolist())

for i, row in df.iterrows():
    print(f"QUESTION: {row['user_input']}")
    print(f"ANSWER: {row['response']}")
    print(f"REFERENCE: {row['reference']}")
    print("=" * 80)

df['question_short'] = df['user_input'].str[:60] + '...'
df[['question_short', 'context_precision', 'context_recall', 'noise_sensitivity(mode=relevant)', 'answer_relevancy','faithfulness']]

['user_input', 'retrieved_contexts', 'response', 'reference', 'context_precision', 'context_recall', 'noise_sensitivity(mode=relevant)', 'answer_relevancy', 'faithfulness']
QUESTION: which cells chordoma develops from?
ANSWER: Chordoma develops from notochord cells that are left behind in the spine or skull after birth. Specifically, these cells can survive beyond fetal development and, due to a change in the gene that carries instructions for making a protein essential for spine formation, may begin to divide too quickly and form chordomas.
REFERENCE: notochord cells
QUESTION: which symtoms will patient have with chordoma compressing spinal cord in thoracic spine?
ANSWER: Based on the provided context, symptoms of chordoma compressing the spinal cord in the thoracic region can include:

- Numbness, tingling, or weakness in the arms or legs
- Pain in the lower back

These symptoms arise due to the compression of nerves by the chordoma tumor.
REFERENCE: Numbness, tingling, or weakness i

,question_short,context_precision,context_recall,noise_sensitivity(mode=relevant),answer_relevancy,faithfulness
0,which cells chordoma develops from?...,1.000000,1.0,0.666667,0.896372,1.000000
1,which symtoms will patient have with chordoma ...,0.125000,1.0,0.333333,0.952421,0.333333
2,why doctors may order bone scan and lungs ct a...,0.000000,1.0,0.000000,0.852357,0.250000
3,which gene problem causes SMA?...,0.407937,1.0,0.000000,0.846111,1.000000
4,what happens with spinal cord without SMN prot...,0.875000,1.0,0.500000,0.805761,1.000000
5,how SMA usually gets diagnosed in USA?...,1.000000,1.0,0.666667,0.640029,0.666667
6,which gene therapy drugs are approved by fda i...,0.833333,1.0,1.000000,0.714387,1.000000
7,which types of spinal stenosis there are?...,1.000000,1.0,0.750000,0.929146,0.750000
8,what usually causes narrowing of spinal canal?...,0.975000,1.0,0.714286,0.784511,0.857143
9,"which symptom comes last, when the most severe...",0.000000,1.0,0.000000,0.870179,0.666667


## Key Findings

**Strengths:**
- `context_recall` = 1.0 across all questions — retriever consistently finds relevant content
- `answer_relevancy` high overall (0.71–0.96) — responses stay on topic
- `faithfulness` = 1.0 on majority of questions — low hallucination rate

**Areas of concern:**
- Questions 2, 9, 12: `noise_sensitivity` = 0.0 — model used irrelevant chunks in answers
- Questions 1, 2: low `faithfulness` — model added information not present in context
- Question 2 (bone scan): `context_precision` = 0.0 — retriever failed to rank relevant chunks first, yet model answered correctly from background knowledge (hallucination with correct outcome — particularly dangerous in medical context)

**Notable observation:**
Ground truth granularity affects scores — brief references penalize complete correct answers.
Clinical expert validation confirmed model accuracy exceeded ground truth completeness in several cases (e.g. SMN1 gene variants).

**Next steps:** see `03_framework_comparison.ipynb` for DeepEval evaluation on the same dataset.